In [23]:
# Select NACA profiles to analyze (list of folder names under repo root)
profiles = [
    'naca0012',
    'naca23012',
    'naca24012',
    'naca2412',
    'naca653-218',
]


In [24]:
# %matplotlib notebook
from matplotlib import pyplot as plt

import pandas as pd
import numpy as np
import os
import re

In [25]:
def extract_cl(fname, tol=1e-5):
    if not os.path.exists(fname):
        print(f'Skipping missing force file: {fname}')
        return np.nan

    try:
        raw = pd.read_csv(fname)
    except pd.errors.EmptyDataError:
        print(f'Skipping empty force file: {fname}')
        return np.nan

    if raw.empty or 'cl_p' not in raw.columns:
        print(f'Skipping malformed force file: {fname}')
        return np.nan

    if 'iter' in raw:
        raw = raw.set_index('iter')

    if 'cl_v' in raw:
        cl = raw['cl_p'] + raw['cl_v']
    else:
        cl = raw['cl_p']

    if cl.empty:
        print(f'Skipping force file with no CL entries: {fname}')
        return np.nan

    cl_last = cl.iloc[-1]

    window_mean = cl.tail(min(len(cl), 20)).mean()
    if abs(cl_last - window_mean) > tol:
        print('Warning, cl is not converged : {}'.format(fname))

    return cl_last

In [26]:
def check_convergence(fname):
    """Check CFD convergence by plotting residuals from stats.csv"""
    stats = pd.read_csv(fname)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.semilogy(stats['iter'], stats['rho'], label='rho residual')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Residual (log scale)')
    ax.set_title('CFD Convergence History')
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)
    
    # Check if converged (residual drops significantly)
    initial_residual = stats['rho'].iloc[0]
    final_residual = stats['rho'].iloc[-1]
    reduction = final_residual / initial_residual
    
    print(f'Initial residual: {initial_residual:.2e}')
    print(f'Final residual: {final_residual:.2e}')
    print(f'Residual reduction: {reduction:.2e}')
    
    if reduction < 1e-4:
        print('✓ Solution appears to be converged')
    else:
        print('⚠ Solution may not be fully converged')
    
    return fig

In [27]:
# Make per-AoA convergence plots for all profiles

aoa_map = {}

for profile in profiles:
    profile_dir = f'../{profile}/runs'
    if not os.path.isdir(profile_dir):
        print(f'Skipping missing profile dir: {profile_dir}')
        continue

    aoa_folders = [d for d in os.listdir(profile_dir) if os.path.isdir(os.path.join(profile_dir, d)) and d.startswith('aoa')]

    # Extract angle values and sort (handle both positive and negative angles)
    aoa_list = []
    for folder in aoa_folders:
        # Match aoa-N for negative angles or aoaN for positive angles
        match = re.match(r'aoa(-?\d+)', folder)
        if match:
            aoa_list.append(int(match.group(1)))
    aoa_list.sort()
    aoa_map[profile] = aoa_list

    print(f'\n{profile}: found {len(aoa_list)} AoA cases -> {aoa_list}')

    convergence_dir = f'../results/{profile}/convergence'
    os.makedirs(convergence_dir, exist_ok=True)

    print(f'Checking convergence for {profile}')
    print('='*60)

    for angle in aoa_list:
        stats_path = f'../{profile}/runs/aoa{angle}/stats.csv'
        if not os.path.exists(stats_path):
            print(f'Skipping missing stats file for AoA {angle}: {stats_path}')
            continue

        try:
            stats = pd.read_csv(stats_path)
        except pd.errors.EmptyDataError:
            print(f'Skipping empty stats file for AoA {angle}: {stats_path}')
            continue

        if stats.empty or 'iter' not in stats or 'rho' not in stats:
            print(f'Skipping malformed stats file for AoA {angle}: {stats_path}')
            continue

        # Handle restarts where iter resets to 1 by stitching cumulative iteration
        iter_series = stats['iter']
        resets = (iter_series.diff() <= 0).fillna(False)
        if resets.any():
            offset = 0
            iter_adj = []
            prev_iter = iter_series.iloc[0]
            iter_adj.append(prev_iter)
            for curr in iter_series.iloc[1:]:
                if curr <= prev_iter:
                    offset += prev_iter
                iter_adj.append(curr + offset)
                prev_iter = curr
            stats['iter_adj'] = iter_adj
            print(f'Note: detected {int(resets.sum())} iteration reset(s) for AoA {angle}; using cumulative iteration for plotting.')
        else:
            stats['iter_adj'] = iter_series

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.semilogy(stats['iter_adj'], stats['rho'], label=f'AoA {angle}°', linewidth=2)

        initial_residual = stats['rho'].iloc[0]
        final_residual = stats['rho'].iloc[-1]
        reduction = final_residual / initial_residual

        status = '✓' if reduction < 1e-4 else '⚠'
        print(f'{status} AoA {angle:2d}°: {initial_residual:.2e} → {final_residual:.2e} (reduction: {reduction:.2e})')

        ax.set_xlabel('Iteration', fontsize=12)
        ax.set_ylabel('Residual (log scale)', fontsize=12)
        ax.set_title(f'CFD Convergence - {profile} - AoA {angle}°', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, which='both', alpha=0.3)
        fig.tight_layout()

        out_path = f'{convergence_dir}/aoa{angle}.png'
        fig.savefig(out_path, dpi=300)
        plt.close(fig)

print('Per-AoA convergence plots saved under each profile/convergence directory.')


naca0012: found 41 AoA cases -> [-20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Checking convergence for naca0012
Note: detected 4 iteration reset(s) for AoA -20; using cumulative iteration for plotting.
⚠ AoA -20°: 1.00e+00 → 7.88e-03 (reduction: 7.88e-03)
Note: detected 3 iteration reset(s) for AoA -19; using cumulative iteration for plotting.
⚠ AoA -19°: 1.00e+00 → 1.42e-02 (reduction: 1.42e-02)
Note: detected 3 iteration reset(s) for AoA -18; using cumulative iteration for plotting.
⚠ AoA -18°: 1.00e+00 → 1.04e-02 (reduction: 1.04e-02)
Note: detected 3 iteration reset(s) for AoA -17; using cumulative iteration for plotting.
⚠ AoA -17°: 1.00e+00 → 9.66e-03 (reduction: 9.66e-03)
Note: detected 3 iteration reset(s) for AoA -16; using cumulative iteration for plotting.
⚠ AoA -16°: 1.00e+00 → 9.38e-03 (reduction: 9.38e-03)
Note: detected 3 iteration reset(s) for AoA -15

In [28]:
# Compute CL curves for all profiles using detected AoA lists
cl_map = {}

for profile, aoa_list in aoa_map.items():
    aoa = np.array(aoa_list)
    valid_aoa = []
    cl = []

    for a in aoa:
        val = extract_cl(f'../{profile}/runs/aoa{a}/force_airfoil.csv', tol=5e-4)
        if pd.notna(val):
            valid_aoa.append(a)
            cl.append(val)

    aoa = np.array(valid_aoa)
    cl = np.array(cl)
    cl_map[profile] = (aoa, cl)


Warning, cl is not converged : ../naca0012/runs/aoa-20/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-19/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-18/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-17/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-16/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-15/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-14/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa-13/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa13/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa14/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa15/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa16/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa17/force_airfoil.csv
Warning, cl is not converged : ../naca0012/runs/aoa18/fo

In [29]:
def extract_cd(fname, tol=1e-5):
    """Extract drag coefficient (CD) from force CSV file."""
    if not os.path.exists(fname):
        print(f'Skipping missing force file: {fname}')
        return np.nan

    try:
        raw = pd.read_csv(fname)
    except pd.errors.EmptyDataError:
        print(f'Skipping empty force file: {fname}')
        return np.nan

    if raw.empty or 'cd_p' not in raw.columns:
        print(f'Skipping malformed force file: {fname}')
        return np.nan

    if 'iter' in raw:
        raw = raw.set_index('iter')

    if 'cd_v' in raw:
        cd = raw['cd_p'] + raw['cd_v']
    else:
        cd = raw['cd_p']

    if cd.empty:
        print(f'Skipping force file with no CD entries: {fname}')
        return np.nan

    cd_last = cd.iloc[-1]

    window_mean = cd.tail(min(len(cd), 20)).mean()
    if abs(cd_last - window_mean) > tol:
        print('Warning, cd is not converged : {}'.format(fname))

    return cd_last


def extract_cm(fname, tol=1e-5):
    """Extract pitching moment coefficient (CM) from force CSV file."""
    if not os.path.exists(fname):
        print(f'Skipping missing force file: {fname}')
        return np.nan

    try:
        raw = pd.read_csv(fname)
    except pd.errors.EmptyDataError:
        print(f'Skipping empty force file: {fname}')
        return np.nan

    if raw.empty or 'cm_p' not in raw.columns:
        print(f'Skipping malformed force file: {fname}')
        return np.nan

    if 'iter' in raw:
        raw = raw.set_index('iter')

    if 'cm_v' in raw:
        cm = raw['cm_p'] + raw['cm_v']
    else:
        cm = raw['cm_p']

    if cm.empty:
        print(f'Skipping force file with no CM entries: {fname}')
        return np.nan

    cm_last = cm.iloc[-1]

    window_mean = cm.tail(min(len(cm), 20)).mean()
    if abs(cm_last - window_mean) > tol:
        print('Warning, cm is not converged : {}'.format(fname))

    return cm_last


# Compute CD curves for all profiles using detected AoA lists
cd_map = {}

for profile, aoa_list in aoa_map.items():
    aoa = np.array(aoa_list)
    valid_aoa = []
    cd = []

    for a in aoa:
        val = extract_cd(f'../{profile}/runs/aoa{a}/force_airfoil.csv', tol=5e-4)
        if pd.notna(val):
            valid_aoa.append(a)
            cd.append(val)

    aoa = np.array(valid_aoa)
    cd = np.array(cd)
    cd_map[profile] = (aoa, cd)

print('CD extraction complete.')


# Compute CM curves for all profiles using detected AoA lists
cm_map = {}

for profile, aoa_list in aoa_map.items():
    aoa = np.array(aoa_list)
    valid_aoa = []
    cm = []

    for a in aoa:
        val = extract_cm(f'../{profile}/runs/aoa{a}/force_airfoil.csv', tol=5e-4)
        if pd.notna(val):
            valid_aoa.append(a)
            cm.append(val)

    aoa = np.array(valid_aoa)
    cm = np.array(cm)
    cm_map[profile] = (aoa, cm)

print('CM extraction complete.')

Warning, cd is not converged : ../naca0012/runs/aoa-20/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-19/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-18/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-17/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-16/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-15/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-14/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa-13/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa13/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa15/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa16/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa18/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa19/force_airfoil.csv
Warning, cd is not converged : ../naca0012/runs/aoa20/fo

In [30]:
# Plot and save CL curves for each profile
for profile, (aoa, cl) in cl_map.items():
    if len(aoa) == 0:
        print(f'{profile}: No valid CL data found. Check force_airfoil.csv files in runs/.')
        continue

    fig, ax = plt.subplots()
    ax.plot(aoa, cl)
    ax.plot(aoa, 2*np.pi*np.deg2rad(aoa + 2.1))
    ax.set_xlabel('Angle of Attack')
    ax.set_ylabel('Lift coefficient')
    ax.legend(['pyBaram', r'Theory ($2\pi\alpha$)'])
    ax.grid()
    fig.savefig(f'../results/{profile}/cl.png', dpi=300)
    plt.close(fig)

print('CL plots saved to each profile directory.')

CL plots saved to each profile directory.


In [31]:
# Plot and save CL vs CD (Drag Polar) for each profile
# Fit only on a filtered "stable" region, but still plot all points (stable + unstable)

for profile in profiles:
    if profile not in cl_map or profile not in cd_map:
        print(f'{profile}: Missing CL or CD data.')
        continue
    
    aoa_cl, cl = cl_map[profile]
    aoa_cd, cd = cd_map[profile]
    
    # Match AoA values between CL and CD
    common_aoa = np.intersect1d(aoa_cl, aoa_cd)
    if len(common_aoa) == 0:
        print(f'{profile}: No common AoA values between CL and CD data.')
        continue
    
    cl_matched = np.array([cl[np.where(aoa_cl == a)[0][0]] for a in common_aoa])
    cd_matched = np.array([cd[np.where(aoa_cd == a)[0][0]] for a in common_aoa])
    
    # Identify a "stable" subset by removing NaNs/Infs and filtering out large outliers
    finite_mask = np.isfinite(cd_matched) & np.isfinite(cl_matched)
    if not finite_mask.any():
        print(f'{profile}: No finite CL/CD pairs after filtering.')
        continue
    
    cd_finite = cd_matched[finite_mask]
    cl_finite = cl_matched[finite_mask]
    
    # Robust outlier filter using median absolute deviation (MAD)
    def mad_filter(arr, thresh=3.5):
        median = np.median(arr)
        mad = np.median(np.abs(arr - median))
        if mad == 0:
            return np.ones_like(arr, dtype=bool)
        z = 0.6745 * (arr - median) / mad
        return np.abs(z) <= thresh
    
    stable_mask = mad_filter(cd_finite) & mad_filter(cl_finite)
    if stable_mask.sum() < 2:
        print(f'{profile}: Not enough stable points for polyfit; skipping fit.')
        cd_stable = cd_finite
        cl_stable = cl_finite
    else:
        cd_stable = cd_finite[stable_mask]
        cl_stable = cl_finite[stable_mask]
    
    # Polynomial fit: CD = f(CL) - fit CL -> CD (standard drag polar form: CD = CD0 + k*CL^2)
    # Then plot with CD on x-axis, CL on y-axis
    if len(cl_stable) >= 2:
        # Fit CD as function of CL (quadratic: CD = a*CL^2 + b*CL + c)
        coeffs = np.polyfit(cl_stable, cd_stable, deg=2)
        poly = np.poly1d(coeffs)
        # Generate smooth CL range for plotting
        cl_fit = np.linspace(cl_stable.min(), cl_stable.max(), 200)
        cd_fit = poly(cl_fit)
    else:
        cl_fit = cl_stable
        cd_fit = cd_stable
    
    # Plot 1: All data + stable subset + polyfit
    fig, ax = plt.subplots()
    # Plot fitting curve first (so it appears on top)
    ax.plot(cd_fit, cl_fit, linewidth=2, color='tab:orange', label='Polyfit (stable)', zorder=7)
    # Then scatter plots
    ax.scatter(cd_matched, cl_matched, marker='o', s=40, zorder=5, color='gray', label='All data')
    ax.scatter(cd_stable, cl_stable, marker='o', s=50, zorder=6, color='tab:blue', label='Stable subset')

    # Axes labeling and limits covering all data (stable + unstable)
    ax.set_xlabel('Drag Coefficient (CD)')
    ax.set_ylabel('Lift Coefficient (CL)')
    finite_cd_all = cd_matched[np.isfinite(cd_matched)]
    finite_cl_all = cl_matched[np.isfinite(cl_matched)]
    if finite_cd_all.size:
        xpad = 0.05 * (finite_cd_all.max() - finite_cd_all.min() + 1e-12)
        ax.set_xlim(finite_cd_all.min() - xpad, finite_cd_all.max() + xpad)
    if finite_cl_all.size:
        ypad = 0.05 * (finite_cl_all.max() - finite_cl_all.min() + 1e-12)
        ax.set_ylim(finite_cl_all.min() - ypad, finite_cl_all.max() + ypad)

    ax.set_title(f'CL vs CD (Drag Polar) - {profile}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(f'../results/{profile}/cl_vs_cd.png', dpi=300)
    plt.close(fig)
    
    # Plot 2: Stable region only + polyfit
    fig, ax = plt.subplots()
    # Plot fitting curve first (so it appears on top)
    ax.plot(cd_fit, cl_fit, linewidth=2, color='tab:orange', label='Polyfit', zorder=7)
    # Then stable scatter plot only
    ax.scatter(cd_stable, cl_stable, marker='o', s=50, zorder=6, color='tab:blue', label='Stable data')

    # Axes labeling and limits covering stable data only
    ax.set_xlabel('Drag Coefficient (CD)')
    ax.set_ylabel('Lift Coefficient (CL)')
    if len(cd_stable) > 0:
        xpad = 0.05 * (cd_stable.max() - cd_stable.min() + 1e-12)
        ax.set_xlim(cd_stable.min() - xpad, cd_stable.max() + xpad)
        ypad = 0.05 * (cl_stable.max() - cl_stable.min() + 1e-12)
        ax.set_ylim(cl_stable.min() - ypad, cl_stable.max() + ypad)

    ax.set_title(f'CL vs CD (Drag Polar - Stable Region) - {profile}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(f'../results/{profile}/cl_vs_cd_stable.png', dpi=300)
    plt.close(fig)

print('CL vs CD plots saved to each profile directory.')

CL vs CD plots saved to each profile directory.


In [32]:
# Plot and save CL/CD (Lift-to-Drag Ratio) vs AoA for each profile
for profile in profiles:
    if profile not in cl_map or profile not in cd_map:
        print(f'{profile}: Missing CL or CD data.')
        continue
    
    aoa_cl, cl = cl_map[profile]
    aoa_cd, cd = cd_map[profile]
    
    # Match AoA values between CL and CD
    common_aoa = np.intersect1d(aoa_cl, aoa_cd)
    if len(common_aoa) == 0:
        print(f'{profile}: No common AoA values between CL and CD data.')
        continue
    
    cl_matched = np.array([cl[np.where(aoa_cl == a)[0][0]] for a in common_aoa])
    cd_matched = np.array([cd[np.where(aoa_cd == a)[0][0]] for a in common_aoa])
    
    # Avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        cl_cd_ratio = np.where(cd_matched != 0, cl_matched / cd_matched, np.nan)
    
    fig, ax = plt.subplots()
    ax.plot(common_aoa, cl_cd_ratio, marker='o', linewidth=2, markersize=6)
    ax.set_xlabel('Angle of Attack (°)')
    ax.set_ylabel('CL/CD (Lift-to-Drag Ratio)')
    ax.set_title(f'CL/CD vs Angle of Attack - {profile}')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    fig.savefig(f'../results/{profile}/cl_cd_ratio.png', dpi=300)
    plt.close(fig)

print('CL/CD vs AoA plots saved to each profile directory.')

CL/CD vs AoA plots saved to each profile directory.


In [33]:
# Plot and save CM vs CL for each profile
# Fit only on a filtered "stable" region, but still plot all points (stable + unstable)

for profile in profiles:
    if profile not in cl_map or profile not in cm_map:
        print(f'{profile}: Missing CL or CM data.')
        continue
    
    aoa_cl, cl = cl_map[profile]
    aoa_cm, cm = cm_map[profile]
    
    # Match AoA values between CL and CM
    common_aoa = np.intersect1d(aoa_cl, aoa_cm)
    if len(common_aoa) == 0:
        print(f'{profile}: No common AoA values between CL and CM data.')
        continue
    
    cl_matched = np.array([cl[np.where(aoa_cl == a)[0][0]] for a in common_aoa])
    cm_matched = np.array([cm[np.where(aoa_cm == a)[0][0]] for a in common_aoa])
    
    # Identify a "stable" subset by removing NaNs/Infs and filtering out large outliers
    finite_mask = np.isfinite(cm_matched) & np.isfinite(cl_matched)
    if not finite_mask.any():
        print(f'{profile}: No finite CL/CM pairs after filtering.')
        continue
    
    cm_finite = cm_matched[finite_mask]
    cl_finite = cl_matched[finite_mask]
    
    # Robust outlier filter using median absolute deviation (MAD)
    def mad_filter(arr, thresh=3.5):
        median = np.median(arr)
        mad = np.median(np.abs(arr - median))
        if mad == 0:
            return np.ones_like(arr, dtype=bool)
        z = 0.6745 * (arr - median) / mad
        return np.abs(z) <= thresh
    
    stable_mask = mad_filter(cm_finite) & mad_filter(cl_finite)
    if stable_mask.sum() < 2:
        print(f'{profile}: Not enough stable points for polyfit; skipping fit.')
        cm_stable = cm_finite
        cl_stable = cl_finite
    else:
        cm_stable = cm_finite[stable_mask]
        cl_stable = cl_finite[stable_mask]
    
    # Polynomial fit: CM = f(CL) (typically quadratic for airfoil moment)
    if len(cl_stable) >= 2:
        coeffs = np.polyfit(cl_stable, cm_stable, deg=2)
        poly = np.poly1d(coeffs)
        cl_fit = np.linspace(cl_stable.min(), cl_stable.max(), 200)
        cm_fit = poly(cl_fit)
    else:
        cl_fit = cl_stable
        cm_fit = cm_stable
    
    # Plot 1: All data + stable subset + polyfit
    fig, ax = plt.subplots()
    ax.plot(cl_fit, cm_fit, linewidth=2, color='tab:orange', label='Polyfit (stable)', zorder=7)
    ax.scatter(cl_matched, cm_matched, marker='o', s=40, zorder=5, color='gray', label='All data')
    ax.scatter(cl_stable, cm_stable, marker='o', s=50, zorder=6, color='tab:blue', label='Stable subset')

    ax.set_xlabel('Lift Coefficient (CL)')
    ax.set_ylabel('Pitching Moment Coefficient (CM)')
    finite_cl_all = cl_matched[np.isfinite(cl_matched)]
    finite_cm_all = cm_matched[np.isfinite(cm_matched)]
    if finite_cl_all.size:
        xpad = 0.05 * (finite_cl_all.max() - finite_cl_all.min() + 1e-12)
        ax.set_xlim(finite_cl_all.min() - xpad, finite_cl_all.max() + xpad)
    if finite_cm_all.size:
        ypad = 0.05 * (finite_cm_all.max() - finite_cm_all.min() + 1e-12)
        ax.set_ylim(finite_cm_all.min() - ypad, finite_cm_all.max() + ypad)

    ax.set_title(f'CM vs CL - {profile}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(f'../results/{profile}/cm_vs_cl.png', dpi=300)
    plt.close(fig)
    
    # Plot 2: Stable region only + polyfit
    fig, ax = plt.subplots()
    ax.plot(cl_fit, cm_fit, linewidth=2, color='tab:orange', label='Polyfit', zorder=7)
    ax.scatter(cl_stable, cm_stable, marker='o', s=50, zorder=6, color='tab:blue', label='Stable data')

    ax.set_xlabel('Lift Coefficient (CL)')
    ax.set_ylabel('Pitching Moment Coefficient (CM)')
    if len(cl_stable) > 0:
        xpad = 0.05 * (cl_stable.max() - cl_stable.min() + 1e-12)
        ax.set_xlim(cl_stable.min() - xpad, cl_stable.max() + xpad)
        ypad = 0.05 * (cm_stable.max() - cm_stable.min() + 1e-12)
        ax.set_ylim(cm_stable.min() - ypad, cm_stable.max() + ypad)

    ax.set_title(f'CM vs CL (Stable Region) - {profile}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(f'../results/{profile}/cm_vs_cl_stable.png', dpi=300)
    plt.close(fig)

print('CM vs CL plots saved to each profile directory.')

naca0012: No common AoA values between CL and CM data.
naca23012: No common AoA values between CL and CM data.
naca24012: No common AoA values between CL and CM data.
naca2412: No common AoA values between CL and CM data.
naca653-218: No common AoA values between CL and CM data.
CM vs CL plots saved to each profile directory.
